# SHEGLAM product exploration

This notebook is intentionally output-free and uses the same validated CSV boundary as `app.py`.
It does not embed scraped results, local Windows/Colab paths, database credentials, or generated images.

**Prerequisite:** provide `sheglam_products.csv` in the repository directory, or set `SHEGLAM_DATA_PATH`.
If the artifact is absent or invalid, execution stops with an actionable error; no rows or metrics are invented.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

def _find_repository_dir() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data_utils.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the repository containing data_utils.py.")


REPOSITORY_DIR = _find_repository_dir()
if str(REPOSITORY_DIR) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIR))
from data_utils import load_products  # noqa: E402

_configured_path = Path(os.getenv("SHEGLAM_DATA_PATH", "sheglam_products.csv")).expanduser()
DATA_PATH = _configured_path if _configured_path.is_absolute() else REPOSITORY_DIR / _configured_path


In [ ]:
df = load_products(DATA_PATH)
print(f"Loaded {len(df):,} rows from {DATA_PATH}")
df.head()


## Data quality and descriptive summaries

Prices and ratings remain source-derived. Invalid values are missing rather than replaced.
This project is descriptive exploration; it does not train a predictive model or claim a
leakage-safe train/test metric.


In [ ]:
quality = pd.Series(
    {
        "rows": len(df),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_price": int(df["Price"].isna().sum()),
        "missing_stars": int(df["Stars"].isna().sum()),
        "missing_best_seller": int(df["Best Seller"].isna().sum()),
    },
    name="value",
)
print(quality.to_string())


In [ ]:
category_summary = (
    df.groupby("Category", observed=True)[["Price", "Stars"]]
    .mean(numeric_only=True)
    .round(2)
    .sort_values("Price", ascending=False, na_position="last")
)
print(category_summary.to_string())


In [ ]:
numeric = df[["Price", "Stars"]].dropna(how="all")
if numeric.empty:
    print("No valid numeric values are available for plots.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(numeric["Price"].dropna(), bins=20, kde=False, ax=axes[0], color="hotpink")
    axes[0].set_title("Price distribution")
    sns.histplot(numeric["Stars"].dropna(), bins=10, kde=False, ax=axes[1], color="mediumpurple")
    axes[1].set_title("Rating distribution")
    fig.tight_layout()
    plt.show()
    plt.close(fig)
